In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import matplotlib.font_manager as fm
import matplotlib

font_path = 'C:\\Windows\\Fonts\\gulim.ttc'
font = fm.FontProperties(fname=font_path).get_name()
matplotlib.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 파일 읽기 / 날씨 데이터와 전력량 데이터 (2015~2024)
weather_df = pd.read_csv('./data/weather_data_201501_202412.csv', encoding='cp949')
electric_df = pd.read_csv('./data/elect_2015_2024.csv',encoding='cp949')

In [ ]:
# unamed 컬럼 제외
electric_df = electric_df.drop(electric_df.columns[0],axis=1)

# 일시에서 월(month)뽑기
weather_df['월'] = pd.to_datetime(weather_df['일시']).dt.month.astype(int)

# 일시, 년(year)으로 교체
weather_df['일시'] =  pd.to_datetime(weather_df['일시']).dt.strftime('%Y').astype(int)

# 결측값 0으로 채우기 
weather_df['최심적설(cm)'] = weather_df['최심적설(cm)'].fillna(0)

# 일시를 연도로 바꾸기 - 뒤에 전력량과 맞춰주기 위하여
weather_df = weather_df.rename(columns={'일시':'연도'})

weather_df


In [ ]:
# 서울시 주택용 전력량 뽑기
seoul_electric_df = electric_df[electric_df['시도'] == '서울특별시']



dfs = []

# 각 연도별로 구별 계약종별 데이터 넣기
for y in range(2014,2025):
      seoul_electric_df_year = seoul_electric_df[seoul_electric_df['연도'] == y]

      seoul_gu_25 = [
          "종로구", "중구", "용산구", "성동구", "동대문구",
          "성북구", "도봉구", "은평구", "서대문구", "마포구",
          "강서구", "구로구", "영등포구", "동작구", "관악구",
          "강남구", "강동구", "송파구", "중랑구", "노원구",
          "서초구", "양천구", "광진구", "강북구", "금천구"
      ]


        # 구별 데이터 선별
      for gu in seoul_gu_25:
            seoul_electric_df_year_gu = seoul_electric_df_year[seoul_electric_df_year['시군구'] == gu]


            dt_type = ['주택용','일반용','교육용','산업용','농사용','가로등','심 야','합 계']


            # 계약별 데이터 선별
            for t in dt_type:
                  seoul_electric_df_year_gu_type = seoul_electric_df_year_gu[seoul_electric_df_year_gu["계약종별"] == t]

                  id_cols = ["연도","시도", "시군구", "계약종별"]
                  month_cols = [ f"{i}월" for i in range(1, 13)]

                    # 멜트해서 id_cols 남기기
                  df_melt = (
                      seoul_electric_df_year_gu_type
                      .reset_index(drop=True)
                      .melt(
                          id_vars=id_cols,
                          value_vars=month_cols,
                          var_name="월",
                          value_name="전력량"
                      )
                  )
                  # 월 숫자만 남기기
                  df_melt["월"] = df_melt["월"].str.replace("월", "").astype(int)
                  dfs.append(df_melt)

# 연도별 구별 월별 데이터 합치기
electric_melt_all = pd.concat(dfs, ignore_index=True)

electric_melt_all

In [ ]:
electric_melt_all

In [ ]:
# 서울시 날씨 데이터 뽑기
weather_seoul_df = weather_df[weather_df['지점명'] == '서울']
weather_seoul_df

In [ ]:
# 날씨값에서 지점, 지점명, 빼기 (중복 or 불필요)
weather_seoul_df_slice = weather_seoul_df.iloc[:,2:]
weather_seoul_df_slice

In [ ]:
electric_melt_all

In [ ]:
# 날씨, 전력량 조인
df = electric_melt_all.merge(weather_seoul_df_slice, on=['연도','월'])
# df_final_2015[df_final_2015['계약종별'] == '심야']


# 날짜로 연도, 월 병합하기
df['날짜'] = df['연도'].astype(str) + '-' + df['월'].astype(str).str.zfill(2)
df_final = df

# 날짜 컬럼 인덱스로 돌리기
# df_final = df.set_index('날짜').drop(columns=['연도','월'])

In [ ]:
electric_melt_all

In [ ]:
df_final

In [ ]:
# 최종 데이터
df_final
df_final.rename(columns={'전력량': '전력사용량'}, inplace=True)

In [ ]:
t = df_final['평균기온(°C)']
df_final['HDD'] = (18 - t).clip(lower=0)
df_final['CDD'] = (t - 24).clip(lower=0)

In [ ]:
df_final.info()

In [ ]:
df_final.drop(['월', '연도'], axis=1, inplace=True)

In [ ]:
df_final_ilban = df_final[df_final['계약종별'] == '일반용']

num_df = df_final_ilban.select_dtypes(include='number')

corr = num_df.corr()
corr_power = corr[['전력사용량']].sort_values(by='전력사용량', ascending=False)

sns.heatmap(corr_power, annot=True, cmap='coolwarm', center=0)
plt.title('서울(일반용)의 전력사용량과 기상요인의 상관계수',fontweight='bold', fontsize=16)
plt.show()

In [ ]:
df_final_ilban= df_final_ilban[['전력량', '평균기온(°C)', '평균현지기압(hPa)',
       '평균해면기압(hPa)', '평균수증기압(hPa)', '평균상대습도(%)', '월합강수량(00~24h만)(mm)',
       '평균풍속(m/s)', '일조율(%)', '최심적설(cm)', '평균지면온도(°C)', 'HDD', 'CDD']]


In [ ]:
# "계약종별"이 "일반용"인 데이터만 필터링
# dfa_final_ilban = df_final[df_final['계약종별'] == '일반용']

# 계약종별과 날짜별로 그룹화하여 전력량 합계 및 다른 변수들의 평균을 계산
# daily_data = df_final_ilban.groupby(['날짜']).agg(
#     전력사용량=('전력량', 'sum'),
#     평균기온=('평균기온(°C)', 'mean'),
#     강수량=('월합강수량(00~24h만)(mm)', 'mean'),
#     습도=('평균상대습도(%)', 'mean'),
#     현지기압=('평균현지기압(hPa)', 'mean'),
#     수증기기압=('평균수증기압(hPa)', 'mean'),
#     풍속=('평균풍속(m/s)', 'mean'),
#     일조율=('일조율(%)', 'mean'),
#     적설=('최심적설(cm)', 'mean'),
#     지면온도=('평균지면온도(°C)', 'mean'),
#     냉방도일=('CDD', 'mean'),
#     난방도일=('HDD', 'mean')
# )

# 상관 관계 계산
corr = df_final_ilban.corr()

# 전력량 합계와 다른 변수들 간의 상관 관계만 추출
corr_power = corr[['전력량']].sort_values(by='전력량', ascending=False)

# 히트맵 시각화
sns.heatmap(corr_power, annot=True, cmap='coolwarm', center=0)
plt.title('서울(일반용)의 전력사용량과 기상요인 상관계수',fontweight='bold', fontsize=16)
plt.show()

In [ ]:
df_final_ilban

In [ ]:
import matplotlib.pyplot as plt

# 날짜에서 년도만 추출
df_final['날짜'] = pd.to_datetime(df_final['날짜'])
df_final['년도'] = df_final['날짜'].dt.year  # 년도만 추출

# 날짜별 전력량 평균 계산
ts = df_final.groupby("날짜")["전력량"].mean()

# 그래프 그리기
plt.figure(figsize=(12, 4))
plt.plot(ts.index, ts.values, label="월 평균 전력량")
plt.title("서울시 전력량(월 평균) 추이")

# X축에 년도만 표시하고, 간격 조정
plt.xticks(ts.index[::12], rotation=45)  # 년도만 표시 (한 번에 12개월마다 년도 표시)

plt.legend(loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
df_final

In [ ]:
import matplotlib.pyplot as plt

# 날짜를 datetime 형식으로 변환하고 년만 추출
df_final['날짜'] = pd.to_datetime(df_final['날짜'])

# 날짜별 평균 전력량 계산
ts = df_final.groupby(df_final['날짜'].dt.year)["전력량"].mean()

# 그래프 그리기
plt.figure(figsize=(12,4))
plt.plot(ts.index, ts.values, label="월 평균 전력량")
plt.title("서울시 전력량(년 평균) 추이")
plt.xticks(ts.index, rotation=45)  # 년만 표시

plt.legend(loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
# 그룹화를 통해 날씨 데이터 영향 비교 전처리 

features = ['평균기온','강수량','습도','현지기압','수증기기압','풍속','일조율','적설','지면온도']

weather_type = (
    df_final
    .groupby(["날짜","계약종별"], as_index=False)
    .agg(
        전력량_평균=("전력량","mean"),
        전력량_합계=("전력량","sum"),
        평균기온=("평균기온(°C)","mean"),
        강수량=("월합강수량(00~24h만)(mm)","mean"),
        습도=("평균상대습도(%)","mean"),
        현지기압=("평균현지기압(hPa)","mean"),
        수증기기압=("평균수증기압(hPa)","mean"),
        풍속=("평균풍속(m/s)","mean"),
        일조율=("일조율(%)","mean"),
        적설=("최심적설(cm)","mean"),
        지면온도=("평균지면온도(°C)","mean"),
        냉방도일=("CDD","mean"),
        난방도일=("HDD","mean")
    )
)

weather = (
    df_final
    .groupby(["날짜"], as_index=False)
    .agg(
        전력량_평균=("전력량","mean"),
        전력량_합계=("전력량","sum"),
        평균기온=("평균기온(°C)","mean"),
        강수량=("월합강수량(00~24h만)(mm)","mean"),
        습도=("평균상대습도(%)","mean"),
        현지기압=("평균현지기압(hPa)","mean"),
        수증기기압=("평균수증기압(hPa)","mean"),
        풍속=("평균풍속(m/s)","mean"),
        일조율=("일조율(%)","mean"),
        적설=("최심적설(cm)","mean"),
        지면온도=("평균지면온도(°C)","mean"),
        냉방도일=("CDD","mean"),
        난방도일=("HDD","mean")
    )
)
weather

In [ ]:
    # 구별, 월별, 종별의 전력량과 날씨 데이터 상관관계 heatmap.version2

corr = weather[
['전력량_합계',"평균기온","강수량","습도",
"현지기압","수증기기압","풍속","일조율","적설","지면온도","냉방도일","난방도일"]
].corr()
corr_power = corr[['전력량_합계']].sort_values(by='전력량_합계', ascending=False)
plt.figure(figsize=(4, 10))
sns.heatmap(corr_power, annot=True, cmap='coolwarm', center=0)
plt.title('전력량 상관계수')
plt.show()

In [ ]:
# 월별 그룹화를 통한 날씨 데이터와 전력량 상관관계 분석 (합계 평균 결과 같음)
corr = weather[
    ['전력량_합계',"평균기온","강수량","습도",
     "현지기압","수증기기압","풍속","일조율","적설","지면온도"]
].corr()

sns.heatmap(corr, annot=True, cmap="coolwarm", center=0)
plt.title("서울시 월별 전력량-날씨 상관관계")
plt.show()


In [ ]:
# 선형 그래프로 날짜 그룹화 + 종별 판단
for idx,name in enumerate(features):
    g = sns.FacetGrid(
        weather_type,
        col="계약종별",
        col_wrap=3,
        height=3,
        sharey=False
    )
    g.map_dataframe(
        sns.regplot,
        x=name,
        y="전력량_합계",
        scatter_kws={"alpha":0.5},
        line_kws={"color":"red"}
    )
    g.fig.suptitle(f"계약종별 {name}-전력량 관계", y=1.03)
    plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 박스플롯 그리기
plt.figure(figsize=(16, 6))
sns.boxplot(x='시군구', y='전력량', data=df_final)
plt.xticks(rotation=45)  # x축 레이블 회전
plt.xlabel('서울시 구별')
plt.ylabel('전력량')
plt.title('서울시 구별 전력량의 분포')
plt.show()


In [ ]:
# 구별 전력량 합계
s = df_final.groupby('시군구')['전력량'].sum().sort_values(ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(s.index, s.values)
plt.title('서울시 구별 전력량 합계')
plt.xlabel('전력량 합계')
plt.ylabel('시군구')
plt.tight_layout()
plt.show()


In [ ]:
# 서울시에서 계약종별의  가장 많은 전력량
s = df_final.groupby('계약종별')['전력량'].mean().sort_values()

plt.Figure(figsize=(8,4))
plt.xlabel('평균 전력량')
plt.title('계약종별 - 평균 전력량')
plt.barh(s.index,s.values)
plt.show()


In [ ]:
# 강남구, 중구, 서초구가 전력량 수요가 많음
plt.figure(figsize=(16,6))
sns.scatterplot(x='시군구', y='전력량', data=df_final)
plt.xticks(rotation=45)
plt.xlabel='서울시 구별'
plt.ylabel='전력량'
plt.show()

In [ ]:
# df_final = df_final.rename({'연도':'years'},axis=1)
# df_final

In [ ]:
df_final_glm= df_final[df_final['계약종별']=='일반용']

In [ ]:
gangwon_industry = df_final_glm[[ '평균기온(°C)',
       '평균현지기압(hPa)', '평균해면기압(hPa)', '평균수증기압(hPa)', '평균상대습도(%)',
       '월합강수량(00~24h만)(mm)', '평균풍속(m/s)', '일조율(%)', '최심적설(cm)', '평균지면온도(°C)',"HDD","CDD",
       '연도', '월', '시도', '시군구', '계약종별', '전력량']]

In [ ]:
gangwon_industry

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. 범주형 변수 변환 (월을 더미 변수로)
# drop_first=True는 다중공선성 방지를 위해 첫 번째 달(1월)을 기준점으로 설정합니다.
df_encoded = pd.get_dummies(gangwon_industry, columns=['월'], drop_first=True, dtype=int)

# 2. 독립변수 리스트 업데이트 (기존 x + 새로 생긴 월 더미 변수들)
month_cols = [col for col in df_encoded.columns if '월' in col]
x_cols = ['평균기온(°C)', '평균현지기압(hPa)', '평균해면기압(hPa)', '평균수증기압(hPa)', 
          '평균상대습도(%)', '월합강수량(00~24h만)(mm)', '평균풍속(m/s)', '일조율(%)', 
          '최심적설(cm)', '평균지면온도(°C)',"HDD","CDD", '연도'] + month_cols
# x_cols = [ '평균기온(°C)', '평균상대습도(%)', '평균현지기압(hPa)', 
#     '평균풍속(m/s)', '일조율(%)', 'years'] + month_cols


# 3. 수치형 변수 표준화 (월 더미 변수는 제외하고 실행)
numeric_cols = ['평균기온(°C)', '평균현지기압(hPa)', '평균해면기압(hPa)', '평균수증기압(hPa)', 
                '평균상대습도(%)', '월합강수량(00~24h만)(mm)', '평균풍속(m/s)', '일조율(%)', 
                '최심적설(cm)', '평균지면온도(°C)',"HDD","CDD",'연도']
# numeric_cols = ['평균기온(°C)', '평균상대습도(%)', '평균현지기압(hPa)', 
#     '평균풍속(m/s)', '일조율(%)', 'years']
scaler = StandardScaler()
df_encoded[numeric_cols] = scaler.fit_transform(df_encoded[numeric_cols])

# 4. 비선형성(제곱항) 추가
df_encoded['평균기온_sq'] = df_encoded['평균기온(°C)']**2

# 5. 모델 설계 및 적합
X = sm.add_constant(df_encoded[x_cols + ['평균기온_sq']])
y = df_encoded['전력량']

df_encoded = pd.get_dummies(gangwon_industry, columns=['월'], drop_first=True, dtype=int)




# 전력 데이터의 특성에 맞춰 Gamma 분포 사용
model = sm.GLM(y, X, family=sm.families.Gamma(link=sm.families.links.log())).fit()

print(model.summary())

In [ ]:
weather_impact = model.params.drop(['const'] + month_cols + ['평균기온_sq'])
weather_impact = weather_impact.sort_values(ascending=False)

plt.figure(figsize=(10, 8))
weather_impact.plot(kind='barh', color='skyblue')
plt.axvline(0, color='red', linestyle='--')
plt.title('Impact of Weather Features on Power Load (Standardized Coefficients)')
plt.xlabel('Coefficient Value (Impact Size)')
plt.ylabel('Weather Features')
plt.tight_layout()
plt.show()

In [ ]:
print(model.pseudo_rsquared())

In [ ]:
import matplotlib.pyplot as plt

# 1) 계약종별로 전력량 합계 계산
ts = df.groupby(['날짜', '계약종별'])['전력량'].sum().unstack('계약종별')

# 2) PeriodIndex면 matplotlib용 timestamp로 변환
if isinstance(ts.index, pd.PeriodIndex):
    ts_plot = ts.copy()
    ts_plot.index = ts_plot.index.to_timestamp()  # PeriodIndex를 Timestamp로 변환
else:
    ts_plot = ts

# 3) 그래프 그리기
plt.figure(figsize=(16,6))

# 계약종별마다 다른 색상으로 선을 그림, label을 설정하여 범례에 표시
for col in ts_plot.columns:
    plt.plot(ts_plot.index, ts_plot[col], marker='o', markersize=2, linewidth=2, label=col)  # label로 계약종별 표시

plt.legend(title='계약종별', bbox_to_anchor=(1.02, 1), loc='upper left')

# 제목, X, Y 레이블 추가
plt.title('계약종별 월별 전력량 추이(서울 전체)', fontsize=16)
plt.xlabel('날짜', fontsize=14)
plt.ylabel('전력량', fontsize=14)

# 범례 추가 (이 부분에서 자동으로 label을 사용)


plt.tight_layout()
plt.show()



In [ ]:
df_final

In [ ]:
sns.boxplot(x='계약종별', y='전력량', data=df_final)
plt.title('서울시 계약종별 전력량')


In [ ]:
sns.boxplot(x='계약종별', y='전력량', data=electric_melt_all)
plt.title('서울시 계약종별 전력사용량')

In [ ]:
# '전력량'을 '전력사용량'으로 변경
df_final.rename(columns={'전력량': '전력사용량'}, inplace=True)

# 변경된 결과 확인
print(df_final.head())


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 박스 플롯 생성
sns.boxplot(x='계약종별', y='전력사용량', data=df_final)

# 제목 추가
plt.title('서울시 계약종별 전력사용량')

# X축 레이블을 '서울'로 변경
  # 여기에서 xlabel을 '서울'로 설정
plt.xlabel('서울')
plt.show()

